In [4]:
from pathlib import Path

import requests
import pandas as pd
import io
import os
import json

# ==============================================================================
# CONFIGURATION FOR MULTIPLE DATASETS
# ==============================================================================
TARGET_DATASETS = [
    {
        "file_name": "OECD_RD_GDP_2000_2025.csv",
        "agency_id": "OECD.STI.STP",
        "dataflow_id": "DSD_MSTI@DF_MSTI",
        "sdmx_query_key": "all",  # Small dataset, safe to download "all" and filter via Pandas
        "filters": {
            "UNIT_MEASURE": "PT_B1GQ",
            "MEASURE": "G"
        }
    },
    {
        "file_name": "OECD_Inflation_CPI_2000_2025.csv",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PRICES@DF_PRICES_ALL",
        "sdmx_query_key": ".A.N.CPI.._T.N.GY",
        "filters": {} 
    },
    {
        "file_name": "OECD_Unemployment_Rate_2000_2025.csv",
        "agency_id": "OECD.ELS.SAE",
        "dataflow_id": "DSD_LFS@DF_LFS_INDIC",
        "sdmx_query_key": "all",  
        "filters": {
            "MEASURE": "UNE_RATE",    # Unemployment Rate
            "UNIT_MEASURE": "PT_LF_SUB", # % of Labour Force
            "SEX": "_T",              # Total (Male + Female)
            "AGE": "_T"               # Total Age
        }
    },
    {
        "file_name": "OECD_Productivity_Variation_2000_2025.csv",
        "agency_id": "OECD.SDD.TPS",
        "dataflow_id": "DSD_PDB@DF_PDB_LV",
        "sdmx_query_key": "all",  
        "filters": {
            "MEASURE": "GDPHRS"  # GDP per hour worked (Labour Productivity)
        }
    },
    {
        "file_name": "OECD_Public_Debt_GDP_2000_2025.csv",
        "agency_id": "OECD.GOV.GIP",
        "dataflow_id": "DSD_GOV@DF_GOV_PF_YU",
        "sdmx_query_key": "A..GGD.PT_B1GQ...",  
        "filters": {}
    }, 
    {
        "file_name": "OECD_Tertiary_Education_2000_2025.csv",
        "agency_id": "OECD.EDU.IMEP",
        "dataflow_id": "DSD_EAG_LSO_EA@DF_LSO_NEAC_DISTR_EA",
        "sdmx_query_key": "._T.Y25T64.ISCED11A_5T8..........OBS...A",  
        "filters": {} # Left empty because the API query key handles all the filtering on the server
    },
    {   "file_name": "OECD_GDP_Growth_2000_2025.csv",
        "agency_id": "OECD.SDD.NAD",
        "dataflow_id": "DSD_NAMAIN1@DF_QNA_EXPENDITURE_GROWTH_OECD",
        "sdmx_query_key": "A.Y..S1.S1.B1GQ._Z._Z._Z.PC.L.GY.T0102",
        "filters": {}  # No Pandas-side filtering needed; server-side key is already precise
    }
]

START_PERIOD = "2000"
END_PERIOD = "2025"
VERSION = "all" # Uses the latest dataset version
PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

headers = {
    "Accept": "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "User-Agent": "DataExtractionScript/4.0 (Python/Requests)"
}

# ==============================================================================
# BATCH EXTRACTION PROCESS
# ==============================================================================

for dataset in TARGET_DATASETS:
    print(f"Processing: {dataset['file_name']}...")
    
    # --------------------------------------------------------------------------
    # FILE EXISTENCE CHECK (SKIP IF ALREADY DOWNLOADED)
    # --------------------------------------------------------------------------
    output_path = DATA_RAW_DIR / dataset['file_name']
    if output_path.exists():
        print(f"  -> File '{dataset['file_name']}' already exists. Skipping download.\n")
        continue

    cwd_path = Path(dataset['file_name'])
    if output_path.exists() or cwd_path.exists():
        print(f"  -> File already exists (either {output_path} or {cwd_path}). Skipping download.\n")
        continue

    # Build the specific URL using the server-side query key to prevent memory overload
    query_key = dataset.get('sdmx_query_key', 'all')
    url = f"https://sdmx.oecd.org/public/rest/data/{dataset['agency_id']},{dataset['dataflow_id']},{VERSION}/{query_key}"
    
    params = {
        "startPeriod": START_PERIOD,
        "endPeriod": END_PERIOD,
        "dimensionAtObservation": "AllDimensions",
        "format": "csvfilewithlabels"
    }
    
    response = requests.get(url, params=params, headers=headers)
    
    if response.status_code == 200:
        csv_stream = io.StringIO(response.text)
        df = pd.read_csv(csv_stream, low_memory=False)
        
        # Protect against completely empty API responses
        if 'OBS_VALUE' not in df.columns:
             print(f"  -> Error: No valid data returned by the API for {dataset['file_name']}.")
             continue
             
        df_clean = df.dropna(subset=['OBS_VALUE']).copy()
        
        # Apply local Pandas filters only if they are defined in the dictionary
        for column_name, filter_value in dataset.get('filters', {}).items():
            if column_name in df_clean.columns:
                df_clean = df_clean[df_clean[column_name] == filter_value]
            else:
                print(f"  -> Warning: Column '{column_name}' not found.")
        
        if df_clean.empty:
            print(f"  -> Error: Dataset is empty after applying filters.")
            
            debug_dict = {}
            for col in df.columns:
                if col not in ['OBS_VALUE', 'TIME_PERIOD', 'REF_AREA']:
                    debug_dict[col] = df[col].dropna().unique().tolist()
            
            with open("DEBUG_OECD_CODES.txt", "w") as f:
                f.write(json.dumps(debug_dict, indent=4))
                
            print("  -> Debug file generated: 'DEBUG_OECD_CODES.txt'. Please check the exact codes.\n")
            continue
        
        target_columns = ['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
        
        if all(col in df_clean.columns for col in target_columns):
            df_tidy = df_clean[target_columns].rename(columns={
                'REF_AREA': 'Country',
                'TIME_PERIOD': 'Year',
                'OBS_VALUE': 'Value'
            })
            
            output_path = DATA_RAW_DIR / dataset['file_name']
            df_tidy = df_tidy.reset_index(drop=True)
            df_tidy.to_csv(output_path, index=False)
            
            print(f"  -> Success! Saved {len(df_tidy)} rows to {output_path}.\n")
        else:
            print(f"  -> Structural error: Missing SDMX columns in {dataset['file_name']}.\n")
    else:
        print(f"  -> HTTP Error {response.status_code} for {dataset['file_name']}.\n")

Processing: OECD_RD_GDP_2000_2025.csv...
  -> File 'OECD_RD_GDP_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Inflation_CPI_2000_2025.csv...
  -> File 'OECD_Inflation_CPI_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Unemployment_Rate_2000_2025.csv...
  -> File 'OECD_Unemployment_Rate_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Productivity_Variation_2000_2025.csv...
  -> File 'OECD_Productivity_Variation_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Public_Debt_GDP_2000_2025.csv...
  -> File 'OECD_Public_Debt_GDP_2000_2025.csv' already exists. Skipping download.

Processing: OECD_Tertiary_Education_2000_2025.csv...
  -> File 'OECD_Tertiary_Education_2000_2025.csv' already exists. Skipping download.

Processing: OECD_GDP_Growth_2000_2025.csv...
  -> File 'OECD_GDP_Growth_2000_2025.csv' already exists. Skipping download.



In [6]:
import requests
import pandas as pd
import io
import os
import itertools
from pathlib import Path

# ==============================================================================
# PATHS — declare all variables before referencing them
# ==============================================================================
PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE_ENERGY = "Eurostat_Energy_Import_Dependency_2000_2025.csv"
OUTPUT_PATH        = DATA_RAW_DIR / OUTPUT_FILE_ENERGY   # ← must come AFTER OUTPUT_FILE_ENERGY

# ==============================================================================
# CONFIGURATION
# ==============================================================================
# To swap the indicator: replace EUROSTAT_DATASET_CODE with the new dataset code.
# Run inspect_eurostat_dimensions(new_code) once to discover the correct
# dimension names and valid filter values before updating FILTERS.
EUROSTAT_DATASET_CODE = "sdg_07_50"

FILTERS = {
    "siec": "TOTAL",  # Energy product dimension (NOT "nrg_bal")
    "unit": "PC",     # Percentage of gross available energy
}

EUROSTAT_BASE    = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
EUROSTAT_HEADERS = {
    "Accept":     "application/json",
    "User-Agent": "DataExtractionScript/4.0 (Python/Requests)"
}

# ==============================================================================
# UTILITY — DIMENSION INSPECTOR (run once for debug)
# ==============================================================================
def inspect_eurostat_dimensions(dataset_code: str) -> None:
    """Prints all dimensions and valid codes for a Eurostat dataset."""
    url    = f"{EUROSTAT_BASE}/{dataset_code}"
    params = {"format": "JSON", "lang": "EN", "lastTimePeriod": 1}
    r = requests.get(url, params=params, headers=EUROSTAT_HEADERS, timeout=60)
    if r.status_code != 200:
        print(f"Error {r.status_code}: {r.text[:300]}")
        return
    d = r.json()
    print(f"\nDimensions of '{dataset_code}':")
    print(f"  Order: {d['id']}")
    for dim_id in d["id"]:
        cats   = d["dimension"][dim_id]["category"]
        labels = cats.get("label", {})
        codes  = sorted(cats["index"], key=lambda c: cats["index"][c])
        print(f"\n  [{dim_id}]")
        for c in codes[:20]:
            print(f"    {c:30s} → {labels.get(c, '')}")
        if len(codes) > 20:
            print(f"    ... ({len(codes) - 20} additional codes)")

# inspect_eurostat_dimensions("sdg_07_50")  # ← uncomment to debug

# ==============================================================================
# EXTRACTION
# ==============================================================================
print(f"Processing: {OUTPUT_FILE_ENERGY}...")

# BUG FIX: check the actual save path (Data/Raw/), not the bare filename
if OUTPUT_PATH.exists():
    print(f"  -> File already exists at '{OUTPUT_PATH}'. Skipping download.\n")
else:
    params = {
        "format":          "JSON",
        "lang":            "EN",
        "sinceTimePeriod": "2000",
        "geoLevel":        "country",  # countries only; excludes EU/EA aggregates
    }
    params.update(FILTERS)

    url      = f"{EUROSTAT_BASE}/{EUROSTAT_DATASET_CODE}"
    response = requests.get(url, params=params, headers=EUROSTAT_HEADERS, timeout=120)

    if response.status_code == 400:
        raise RuntimeError(
            f"Eurostat 400 Bad Request.\n"
            f"Detail: {response.text}\n"
            f"Hint: uncomment inspect_eurostat_dimensions() to verify the correct "
            f"dimension names and filter codes for this dataset."
        )
    if response.status_code == 413:
        raise RuntimeError(
            "Eurostat 413: asynchronous response. Wait ~2 minutes and re-run the cell."
        )
    if response.status_code != 200:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")

    # --- PARSE JSON-STAT 2.0 ---
    data   = response.json()
    dims   = data["dimension"]
    size   = data["size"]
    ids    = data["id"]
    values = data["value"]

    dim_categories = []
    for dim_id in ids:
        cats          = dims[dim_id]["category"]
        ordered_codes = sorted(cats["index"], key=lambda c: cats["index"][c])
        dim_categories.append(ordered_codes)

    records = []
    for combo in itertools.product(*[range(s) for s in size]):
        flat_idx = 0
        stride   = 1
        for i in range(len(size) - 1, -1, -1):
            flat_idx += combo[i] * stride
            stride   *= size[i]

        val = (
            values.get(str(flat_idx))
            if isinstance(values, dict)
            else (values[flat_idx] if flat_idx < len(values) else None)
        )
        if val is None:
            continue

        row = {ids[i]: dim_categories[i][combo[i]] for i in range(len(ids))}
        row["OBS_VALUE"] = float(val)
        records.append(row)

    if not records:
        raise ValueError(
            "No data after parsing. "
            "Check FILTERS with inspect_eurostat_dimensions()."
        )

    df_raw = pd.DataFrame(records)

    df_tidy = (
        df_raw[["geo", "time", "OBS_VALUE"]]
        .rename(columns={
            "geo":       "Country",  # ISO 2 country code (Eurostat standard)
            "time":      "Year",
            "OBS_VALUE": "Value"     # % of gross available energy
        })
        .sort_values(["Country", "Year"])
        .reset_index(drop=True)
    )

    df_tidy.to_csv(OUTPUT_PATH, index=False)
    print(f"  -> Success! {len(df_tidy)} rows saved to '{OUTPUT_PATH}'.")
    print(f"\nPreview:\n{df_tidy.head(10)}\n")

Processing: Eurostat_Energy_Import_Dependency_2000_2025.csv...
  -> File already exists at 'c:\Documenti\UNIMIB\Data Science Lab\PROJECT\Data\Raw\Eurostat_Energy_Import_Dependency_2000_2025.csv'. Skipping download.



In [16]:
import requests
import pandas as pd
import io
from pathlib import Path

# ==============================================================================
# CONFIGURATION
# Each OECD "Government at a Glance" edition is a separate dataflow.
# Historical depth requires fetching all editions and stacking them.
# Priority order matters for deduplication: higher index = newer = preferred.
# ==============================================================================
PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = "OECD_Trust_Institutions_2000_2025.csv"
OUTPUT_PATH = DATA_RAW_DIR / OUTPUT_FILE

START_PERIOD = "2000"
END_PERIOD   = "2025"
MEASURE_CODE = "TRUST_NG"   # Trust in national government
                             # Replace with any code from the MEASURE inspection
                             # to download a different trust indicator.

HEADERS_SDMX = {
    "Accept":     "application/vnd.sdmx.data+csv; charset=utf-8; version=2",
    "User-Agent": "DataExtractionScript/4.0 (Python/Requests)"
}

# All confirmed dataflows from the catalog search, oldest → newest.
# Deduplication keeps the newest edition for any overlapping country/year.
EDITIONS = [
    {
        "label":       "Gov@Glance 2023 — Trust and Democratic Governance",
        "agency_id":   "OECD.GOV.GIP",
        "dataflow_id": "DSD_GOV_TDG_SPS_GPC@DF_GOV_TDG_2023",
        "version":     "1.0",
    },
    {
        "label":       "Gov@Glance 2023 — Trust, Satisfaction and Governance Cycle",
        "agency_id":   "OECD.GOV.GIP",
        "dataflow_id": "DSD_GOV_TDG_SPS_GPC@DF_GOV_TDG_SPS_GPC_2023",
        "version":     "1.0",
    },
    {
        "label":       "Gov@Glance 2025 — Trust, Security and Dignity (v1.1)",
        "agency_id":   "OECD.GOV.GIP",
        "dataflow_id": "DSD_GOV_INT@DF_GOV_TDG_2025",
        "version":     "1.1",
    },
]

# ==============================================================================
# EXTRACTION — fetch all editions and stack
# ==============================================================================
print(f"Processing: {OUTPUT_FILE}...")

if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print("  -> Previous file deleted.")

all_frames = []

for edition in EDITIONS:
    label = edition["label"]
    print(f"\n  -> Fetching: '{label}'...")

    url = (
        f"https://sdmx.oecd.org/public/rest/data/"
        f"{edition['agency_id']},{edition['dataflow_id']},{edition['version']}/all"
    )
    params = {
        "startPeriod":            START_PERIOD,
        "endPeriod":              END_PERIOD,
        "dimensionAtObservation": "AllDimensions",
        "format":                 "csvfilewithlabels"
    }

    try:
        response = requests.get(url, params=params, headers=HEADERS_SDMX, timeout=120)
    except requests.exceptions.Timeout:
        print(f"     Timeout. Skipping this edition.")
        continue

    if response.status_code != 200:
        print(f"     HTTP {response.status_code}. Skipping this edition.")
        continue

    df_raw = pd.read_csv(io.StringIO(response.text), low_memory=False)

    if "OBS_VALUE" not in df_raw.columns or df_raw.empty:
        print(f"     Empty or malformed response. Skipping.")
        continue

    # Filter to the target measure
    if "MEASURE" not in df_raw.columns:
        print(f"     MEASURE column missing. Skipping.")
        continue

    df_filtered = df_raw[df_raw["MEASURE"] == MEASURE_CODE].dropna(subset=["OBS_VALUE"]).copy()

    if df_filtered.empty:
        available = df_raw["MEASURE"].dropna().unique().tolist()
        print(f"     MEASURE='{MEASURE_CODE}' not found. Available: {available}. Skipping.")
        continue

    # Keep only PC_POP unit if multiple units are present
    if "UNIT_MEASURE" in df_filtered.columns:
        units = df_filtered["UNIT_MEASURE"].unique().tolist()
        if len(units) > 1:
            print(f"     Multiple units detected: {units}. Keeping PC_POP.")
            df_filtered = df_filtered[df_filtered["UNIT_MEASURE"] == "PC_POP"]

    df_edition = (
        df_filtered[["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]]
        .rename(columns={
            "REF_AREA":    "Country",
            "TIME_PERIOD": "Year",
            "OBS_VALUE":   "Value"
        })
        .copy()
    )

    print(f"     {len(df_edition)} rows for MEASURE='{MEASURE_CODE}'.")
    all_frames.append(df_edition)

if not all_frames:
    raise RuntimeError("All editions failed. Re-run the catalog search to check IDs.")

# ==============================================================================
# MERGE — stack all editions, keep newest value for any country/year overlap
# ==============================================================================
# Concatenation order is oldest → newest (defined in EDITIONS list above).
# drop_duplicates with keep="last" therefore retains the newest edition's value.
df_combined = (
    pd.concat(all_frames, ignore_index=True)
    .drop_duplicates(subset=["Country", "Year"], keep="last")
    .sort_values(["Country", "Year"])
    .reset_index(drop=True)
)

df_combined["Value"] = pd.to_numeric(df_combined["Value"], errors="coerce")
df_combined = df_combined.dropna(subset=["Value"])

print(f"\n  -> Combined total: {len(df_combined)} unique country/year rows.")
print(f"     Years present:   {sorted(df_combined['Year'].unique().tolist())}")
print(f"     Countries:       {df_combined['Country'].nunique()}")

df_combined.to_csv(OUTPUT_PATH, index=False)
print(f"\n  -> Saved to '{OUTPUT_PATH}'.")
print(f"\nPreview:\n{df_combined.head(10)}\n")

Processing: OECD_Trust_Institutions_2000_2025.csv...

  -> Fetching: 'Gov@Glance 2023 — Trust and Democratic Governance'...
     21 rows for MEASURE='TRUST_NG'.

  -> Fetching: 'Gov@Glance 2023 — Trust, Satisfaction and Governance Cycle'...
     21 rows for MEASURE='TRUST_NG'.

  -> Fetching: 'Gov@Glance 2025 — Trust, Security and Dignity (v1.1)'...
     208 rows for MEASURE='TRUST_NG'.

  -> Combined total: 52 unique country/year rows.
     Years present:   [2021, 2023]
     Countries:       33

  -> Saved to 'c:\Documenti\UNIMIB\Data Science Lab\PROJECT\Data\Raw\OECD_Trust_Institutions_2000_2025.csv'.

Preview:
  Country  Year  Value
0     AUS  2021   38.0
1     AUS  2023   38.1
2     AUT  2021   25.8
3     BEL  2021   31.8
4     BEL  2023   36.3
5     CAN  2021   37.5
6     CAN  2023   34.6
7     CHE  2023   23.6
8     CHL  2023   53.7
9     COL  2021   20.5



In [ ]:
import requests
import json

HEADERS = {"Accept": "application/json", "User-Agent": "DataExtractionScript/4.0"}

r = requests.get(
    "https://stats.oecd.org/sdmx-json/data/BLI/all/OECD",
    params={"startTime": "2020", "endTime": "2021",
            "dimensionAtObservation": "allDimensions"},
    headers=HEADERS, timeout=60
)
payload = r.json()

def print_structure(obj, indent=0, max_depth=4, max_list_items=2):
    """Recursively prints the shape of a JSON object without printing data values."""
    prefix = "  " * indent
    if indent > max_depth:
        print(f"{prefix}...")
        return
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(v, (dict, list)):
                print(f"{prefix}{k}:")
                print_structure(v, indent+1, max_depth, max_list_items)
            else:
                # Truncate scalar values for readability
                print(f"{prefix}{k}: {str(v)[:80]}")
    elif isinstance(obj, list):
        print(f"{prefix}[list of {len(obj)} items]")
        for item in obj[:max_list_items]:
            print_structure(item, indent+1, max_depth, max_list_items)
        if len(obj) > max_list_items:
            print(f"{prefix}  ...")

print_structure(payload)

=== BLI DIMENSIONS (0 found) ===

